### 1. Extract Folder Topic Names

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import os
import re

In [2]:
dataset_dir = Path('./20_newsgroups')
topic_names = sorted([topic.name for topic in dataset_dir.iterdir() if topic.is_dir()])
print(f"Number of topics: {len(topic_names)}")

Number of topics: 20


### 2. Data Loading Pipeline

In [3]:
def clean_newsgroup_body(raw_text):
    parts = raw_text.split('\n\n', 1) # headers from body
    body = parts[1] if len(parts) > 1 else raw_text
    text = body.lower()
    text = re.sub(r'^\s*[|>#]{2,}\s*$', '', text, flags=re.MULTILINE) # remove lines starting with >, |, # (common in quoted text)
    text = re.sub(r'(^|\n)\s*>[^\n]*', '', text) # remove quoted text, > & spaces
    text = re.sub(r'[^a-z\s]', ' ', text) # remove non-alphabetic characters, keep spaces
    text = re.sub(r'\s+', ' ', text).strip() # remove extra spaces and newlines, replace with single space
    text = re.sub(r'\S*@\S*\s?', '', text) # remove email addresses
    text = re.sub(r'\s+', ' ', text).strip() # removes white spaces and newlines, replaces with single space
    return text

In [4]:
def load_dataset_to_dataframe(folder_path):
    data_records = []
    
    for label_dir in os.listdir(folder_path): # iterate through root newsgroup folders
        dir_path = os.path.join(folder_path, label_dir)
        
        if not os.path.isdir(dir_path): # skipping non-directory files if any
            continue
            
        for file_name in os.listdir(dir_path): # iterate through files in each category folder
            file_path = os.path.join(dir_path, file_name)
            try:
                with open(file_path, "r", encoding="latin-1") as f: # read and use latin-1 encoding to prevent UnicodeDecodeErrors
                    raw_content = f.read()
                    cleaned_text = clean_newsgroup_body(raw_content) # clean the text using the defined function
                    if cleaned_text:
                        data_records.append({"file_id": file_name, "label": label_dir, 
                        "text": cleaned_text, "original_length": len(raw_content)}) # append only cleaned text
            except Exception as e:
                print(f"Failed to read {file_path}: {e}")
                
    return pd.DataFrame(data_records) # convert dict list of records to a DataFrame

In [5]:
dataset_path = "./20_newsgroups" 
df = load_dataset_to_dataframe(dataset_path)

print(f"Successfully loaded {len(df)} documents.")
print("\nClass Distribution:")
print(df['label'].value_counts())

Successfully loaded 19955 documents.

Class Distribution:
label
talk.politics.guns          1000
talk.politics.mideast       1000
talk.politics.misc          1000
talk.religion.misc          1000
alt.atheism                  999
rec.sport.baseball           999
rec.sport.hockey             999
sci.crypt                    999
sci.electronics              999
sci.space                    999
comp.windows.x               998
rec.autos                    998
sci.med                      998
comp.graphics                997
comp.sys.ibm.pc.hardware     997
rec.motorcycles              997
soc.religion.christian       997
comp.sys.mac.hardware        996
comp.os.ms-windows.misc      992
misc.forsale                 991
Name: count, dtype: int64


In [6]:
df.shape

(19955, 4)

In [7]:
df.head()

,file_id,label,text,original_length
0,49960,alt.atheism,archive name atheism resources alt atheism arc...,12425
1,51060,alt.atheism,archive name atheism introduction alt atheism ...,32531
2,51119,alt.atheism,in article mimsy umd edu mangoe cs umd edu cha...,4550
3,51120,alt.atheism,dmn kepler unh edu until kings become philosop...,2067
4,51121,alt.atheism,in article n hy apr harder ccr p ida org n hy ...,1347


In [8]:
random_texts = df['text'].sample(n=10)

for i, text in enumerate(random_texts, 1):
    print(f"{i}) - RANDOM DOCUMENT")
    print(text + "...\n")

1) - RANDOM DOCUMENT
in article schinder leprss gsfc nasa gov paul j schinder schinder leprss gsfc nasa gov wrote when you look into the math of the different possible frames of reference you will find out that this time is not physically relevant the relevant time for the falling object center of the hole where the evil singularity lurks there is no singularity at the horizon the appearance that there is one comes from attempting to use a coordinate system that doesn t map properly physically in this geometry rather analogous to the singularity at the north and south poles of the earth when using latitude and longitude lines such singular behavior as the earlier post suggested infinite time to reach the horizon is a mathematical artifact of the wrong choice of coordinates as paul says in the object s frame of reference it will hit the horizon in finite time and then plunge to the singularity that is a real singularity in a short for small holes further finite amount of time not only t

### 3. Text Pre-Processing Stage

In [9]:
import nltk
import ssl
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from tqdm import tqdm

In [10]:
try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

nltk.download('punkt_tab')       # tokenizer
nltk.download('stopwords')   # stop words list
nltk.download('wordnet')     # lemmatization dictionary
nltk.download('omw-1.4')     # open multilingual wordnet (required by newer NLTK versions)

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\manoj\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\manoj\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\manoj\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\manoj\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [11]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

In [12]:
custom_stop_words = {'write', 'article', 'post', 'know', 'like', 'just', 'say', 'think', 'use', 'make', 'good', 'time', 'people'}
stop_words.update(custom_stop_words)

In [13]:
def nltk_process_text(text):
    tokens = word_tokenize(text) # tokenize the text (splits string into list of words)
    cleaned_tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words and len(word) > 2]
    return " ".join(cleaned_tokens)

In [14]:
tqdm.pandas(desc="NLTK Tokenizing, Lemmatizing & Stop Word Removal")

print("Starting NLTK pre-processing...")
df['processed_text'] = df['text'].progress_apply(nltk_process_text)

print("Pre-processing complete! Check the first few rows:")
display(df[['label', 'processed_text']].head())

Starting NLTK pre-processing...


NLTK Tokenizing, Lemmatizing & Stop Word Removal: 100%|██████████| 19955/19955 [00:15<00:00, 1315.54it/s]

Pre-processing complete! Check the first few rows:


,label,processed_text
0,alt.atheism,archive name atheism resource alt atheism arch...
1,alt.atheism,archive name atheism introduction alt atheism ...
2,alt.atheism,mimsy umd edu mangoe umd edu charley wingate w...
3,alt.atheism,dmn kepler unh edu king become philosopher phi...
4,alt.atheism,apr harder ccr ida org harder ccr ida org bob ...


In [15]:
df[['file_id', 'original_length', 'text', 'processed_text', 'label']].head()

,file_id,original_length,text,processed_text,label
0,49960,12425,archive name atheism resources alt atheism arc...,archive name atheism resource alt atheism arch...,alt.atheism
1,51060,32531,archive name atheism introduction alt atheism ...,archive name atheism introduction alt atheism ...,alt.atheism
2,51119,4550,in article mimsy umd edu mangoe cs umd edu cha...,mimsy umd edu mangoe umd edu charley wingate w...,alt.atheism
3,51120,2067,dmn kepler unh edu until kings become philosop...,dmn kepler unh edu king become philosopher phi...,alt.atheism
4,51121,1347,in article n hy apr harder ccr p ida org n hy ...,apr harder ccr ida org harder ccr ida org bob ...,alt.atheism


In [16]:
output_dir = "./data"
output_file = os.path.join(output_dir, "processed_newsgroups.csv")

os.makedirs(output_dir, exist_ok=True)

print(f"Saving processed data to {output_file}...")
df[['file_id', 'original_length', 'text', 'processed_text', 'label']].to_csv(output_file, index=False) 
print("Save complete! Your intermediate data is safely stored.")

Saving processed data to ./data\processed_newsgroups.csv...
Save complete! Your intermediate data is safely stored.
